In [ ]:
import spacy

nlp = spacy.load("en_core_web_trf")

In [2]:
aux_token = ["aren", "wer", "weren", "wasn", "doesn", "didn", "couldn", "shan", "cannot", "didnt", "dont", "don", "youre", "im", "theyre"]

def in_aux_token(token):
    return token.lower() in aux_token

In [3]:
def decode_hidden_message(prompt, text, hidden_message, nlp,
                          encode_bit_1, encode_bit_0, neutral_pos):
    full_text = prompt + text
    doc = nlp(full_text)
    prompt_len = len(prompt)
    exploded = []
    i = 0
    
    while i < len(doc):
        tok = doc[i]
        
        if tok.text.lower() == "can" and i+1 < len(doc) and doc[i+1].text.lower() == "not":
            exploded.append(("cannot", "VERB", tok.idx))
            i += 2
            continue
        
        if i+1 < len(doc) and doc[i+1].text in {"n't", "n’t"}:
            base = tok.text + "n"
            exploded.append((base, tok.pos_, tok.idx))
            exploded.append(("'t", "PART", doc[i+1].idx))
            i += 2
            continue
            
        if tok.text.lower() in {"you", "they"}:
            if i+1 < len(doc) and doc[i+1].text.lower() in {"re"}:
                merged = tok.text + doc[i+1].text
                exploded.append((merged, "VERB", tok.idx))
                i += 2
                continue
        
        exploded.append((tok.text, tok.pos_, tok.idx))
        i += 1

    decoded = ""
    last_pos = None
    last_text = ""
    print("Decoding tokens:")
    for word, pos, idx in exploded:
        if in_aux_token(word):
            pos = "VERB"
        if idx < prompt_len:
            last_pos, last_text = pos, word
            continue
            
        if len(decoded) == len(hidden_message):
            break

        if last_pos == "PROPN" and pos == "PROPN" \
           and last_text[:1].isupper() and word[:1].isupper():
            print(f"Word: '{word}', POS: {pos} -> Skipped (NER‑style)")
        elif word in {"'s", "’s", "'t", "’t"}:
            print(f"Word: '{word}', POS: {pos} -> Skipped (contraction)")
        elif pos in encode_bit_0:
            decoded += "0"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 0")
        elif pos in encode_bit_1:
            decoded += "1"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 1")
        else:
            print(f"Word: '{word}', POS: {pos} -> Skipped (neutral/other)")
        last_pos, last_text = pos, word

    print(f"\nD: {decoded}")
    print(f"H: {hidden_message}")
    if decoded == hidden_message:
        print("Decoded message matches!")
        return decoded, True
    else:
        print("Mismatch.")
        return decoded, False

In [4]:
encode_bit_1 = ["VERB", "ADP", "SCONJ", "DET", "PRON", "ADJ", "PUNCT"]
encode_bit_0 = ["ADV", "AUX", "PART", "PROPN", "NOUN", "CCONJ", "NUM"]
neutral_pos = ["INTJ", "SYM"]

In [6]:
decode_hidden_message(
    "Renewable energy sources like solar and wind power are essential for reducing greenhouse gas emissions and combating climate change.",
    " However, these sources are often intermittent, and their variable output makes it challenging for utilities, industrial, and even individual residential consumers to rely on them as a primary source of electricity.",
    "011000110110111101110011", nlp, encode_bit_1, encode_bit_0, neutral_pos)

Decoding tokens:
Word: 'However', POS: ADV -> Decoded as 0
Word: ',', POS: PUNCT -> Decoded as 1
Word: 'these', POS: DET -> Decoded as 1
Word: 'sources', POS: NOUN -> Decoded as 0
Word: 'are', POS: AUX -> Decoded as 0
Word: 'often', POS: ADV -> Decoded as 0
Word: 'intermittent', POS: ADJ -> Decoded as 1
Word: ',', POS: PUNCT -> Decoded as 1
Word: 'and', POS: CCONJ -> Decoded as 0
Word: 'their', POS: PRON -> Decoded as 1
Word: 'variable', POS: ADJ -> Decoded as 1
Word: 'output', POS: NOUN -> Decoded as 0
Word: 'makes', POS: VERB -> Decoded as 1
Word: 'it', POS: PRON -> Decoded as 1
Word: 'challenging', POS: ADJ -> Decoded as 1
Word: 'for', POS: SCONJ -> Decoded as 1
Word: 'utilities', POS: NOUN -> Decoded as 0
Word: ',', POS: PUNCT -> Decoded as 1
Word: 'industrial', POS: ADJ -> Decoded as 1
Word: ',', POS: PUNCT -> Decoded as 1
Word: 'and', POS: CCONJ -> Decoded as 0
Word: 'even', POS: ADV -> Decoded as 0
Word: 'individual', POS: ADJ -> Decoded as 1
Word: 'residential', POS: ADJ -> Dec

('011000110110111101110011', True)